# Matrix Optimizer - Basis Randomization Edition

**Key**: Random change-of-basis each trial to escape Laderman local minimum.
**Vocab**: Extended to {-2,-1,0,1,2}.
**Schedule**: 50 trials x 20k epochs, 4-phase training.

In [1]:
import numpy as np
import torch, torch.nn.functional as F
import time, os, json
from collections import defaultdict, Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VOCAB = [-2,-1,0,1,2]
VOCAB_T = torch.tensor(VOCAB, dtype=torch.float32, device=DEVICE)

class Config:
    N_TRIALS = 50
    EPOCHS = 20000
    # Modified Phases: Micro-Jolt added to Phase 4, Softened L1 in Phase 5
    # Modified Phases: Pure greedy descent in Phase 5
    PHASES = [
        (0, 3000, 0.15, 0.01, 0.0),            # Escaper
        (3000, 8000, 0.08, 0.005, 1e-5),       # Descent 1
        (8000, 14000, 0.02, 0.001, 1e-4),      # Descent 2
        (14000, 18000, 0.001, 0.0005, 1e-4),   # Micro-Jolt
        (18000, 20000, 0.0, 0.002, 0.0)        # FREE-FALL: Zero noise, zero L1, HIGH LR (0.002)
    ]
    NOISE_INIT = 0.15
    PLATEAU_PATIENCE = 500
    PLATEAU_BURST = 0.05
    VERBOSE_INTERVAL = 500
    WIN_THRESHOLD = 1e-8

cfg = Config()
print(f"{cfg.N_TRIALS} trials x {cfg.EPOCHS} epochs, vocab={VOCAB}")
print("Config loaded")

50 trials x 20000 epochs, vocab=[-2, -1, 0, 1, 2]
Config loaded


In [2]:
def load_data(path="factorizations_r.npz"):
    d = np.load(path, allow_pickle=True)
    t = d["3,3,3"]
    if t.ndim == 0: t = t.item()
    if isinstance(t,dict): U,V,W = t["u"],t["v"],t["w"]
    else: U,V,W = t
    return np.array(U,dtype=np.float32), np.array(V,dtype=np.float32), np.array(W,dtype=np.float32)

U_23,V_23,W_23 = load_data()
print(f"Loaded: U {U_23.shape}")

def build_target():
    T = torch.zeros((9,9,9), dtype=torch.float32, device=DEVICE)
    for i in range(3):
        for j in range(3):
            for k in range(3):
                T[i*3+j, j*3+k, i*3+k] = 1.0
    return T

T_target = build_target()
print(f"Target built: {T_target.shape}")

Loaded: U (9, 23)
Target built: torch.Size([9, 9, 9])


In [3]:
def random_orthogonal(dim):
    H = torch.randn(dim, dim, device=DEVICE)
    Q, R = torch.linalg.qr(H)
    return Q * torch.sign(torch.diag(R)).unsqueeze(0)

def apply_basis(T, P, Q, R):
    T1 = torch.einsum("ia,abc->ibc", P, T)
    T2 = torch.einsum("jb,ibc->ijc", Q, T1)
    T3 = torch.einsum("kc,ijc->ijk", R, T2)
    return T3

P,Q,R = random_orthogonal(9),random_orthogonal(9),random_orthogonal(9)
Tt = apply_basis(T_target,P,Q,R)
Tb = apply_basis(Tt,P.T,Q.T,R.T)
print(f"Round-trip MSE: {F.mse_loss(Tb,T_target).item():.2e}")
del P,Q,R,Tt,Tb
print("Basis randomization ready")

Round-trip MSE: 8.56e-15
Basis randomization ready


In [4]:
def get_params(epoch):
    for s,e,n,lr,l1 in cfg.PHASES:
        if s <= epoch < e:
            frac = (epoch-s)/(e-s) if e>s else 0.0
            return n*(1-0.5*frac), lr, l1
    return 0.0, 1e-4, cfg.PHASES[-1][4]

def run_trial(tid, col_drop, verbose=True):
    start_t = time.time()
    P,Q,R = random_orthogonal(9),random_orthogonal(9),random_orthogonal(9)
    T_trans = apply_basis(T_target,P,Q,R)
    keep = [i for i in range(23) if i != col_drop]
    U_init = torch.tensor(U_23[:,keep], dtype=torch.float32, device=DEVICE)
    V_init = torch.tensor(V_23[:,keep], dtype=torch.float32, device=DEVICE)
    W_init = torch.tensor(W_23[:,keep], dtype=torch.float32, device=DEVICE)
    
    U = P @ U_init + cfg.NOISE_INIT*torch.randn(9,22,device=DEVICE)
    V = Q @ V_init + cfg.NOISE_INIT*torch.randn(9,22,device=DEVICE)
    W = R @ W_init + cfg.NOISE_INIT*torch.randn(9,22,device=DEVICE)
    U.requires_grad_(True); V.requires_grad_(True); W.requires_grad_(True)
    
    opt = torch.optim.Adam([U,V,W], lr=0.01)
    best_loss = float("inf")
    best_state = None; patience = 0
    
    for ep in range(cfg.EPOCHS):
        nmag,lr_curr,l1s = get_params(ep)
        for g in opt.param_groups: g["lr"] = lr_curr
        opt.zero_grad()
        
        mse = F.mse_loss(torch.einsum("ir,jr,kr->ijk",U,V,W), T_trans)
        (mse + l1s*(U.abs().mean()+V.abs().mean()+W.abs().mean())).backward()
        opt.step()
        
        if nmag > 0:
            with torch.no_grad():
                U.add_(torch.randn_like(U)*nmag)
                V.add_(torch.randn_like(V)*nmag)
                W.add_(torch.randn_like(W)*nmag)
                
        cv = mse.item()
        if cv < best_loss:
            best_loss = cv
            best_state = (U.detach().clone(),V.detach().clone(),W.detach().clone(),P,Q,R)
            patience = 0
        else: 
            patience += 1
            
        if patience > cfg.PLATEAU_PATIENCE and ep > cfg.PHASES[1][0]:
            with torch.no_grad():
                U.add_(torch.randn_like(U)*cfg.PLATEAU_BURST)
                V.add_(torch.randn_like(V)*cfg.PLATEAU_BURST)
                W.add_(torch.randn_like(W)*cfg.PLATEAU_BURST)
            patience = 0
            
        if verbose and ep % cfg.VERBOSE_INTERVAL == 0:
            print(f"  T{tid:2d} Ep{ep:5d} MSE:{cv:.2e} Noi:{nmag:.3f}")
            
        # --- THE EXACT MATCH CALLBACK START ---
        if cv < cfg.WIN_THRESHOLD:
            if verbose: 
                print(f"\n  *** E A R L Y   S U C C E S S *** ep{ep}! MSE: {cv:.2e}")
            out_dir = "outputs"
            if not os.path.exists(out_dir): os.makedirs(out_dir)
            
            # Map weights back to standard basis before saving
            with torch.no_grad():
                U_exact = P.T @ U.detach()
                V_exact = Q.T @ V.detach()
                W_exact = R.T @ W.detach()
            
            match_path = os.path.join(out_dir, f"EXACT_MATCH_T{tid}_Col{col_drop}.pt")
            torch.save({
                "U": U_exact.cpu(), 
                "V": V_exact.cpu(), 
                "W": W_exact.cpu(), 
                "mse": cv, 
                "epoch": ep
            }, match_path)
            
            if verbose: 
                print(f"  [SAVED] Golden tensor dumped to {match_path}\n")
            break
        # --- THE EXACT MATCH CALLBACK END ---

    U_b,V_b,W_b,Pb,Qb,Rb = best_state
    with torch.no_grad():
        U_o = Pb.T @ U_b; V_o = Qb.T @ V_b; W_o = Rb.T @ W_b
        mse_o = F.mse_loss(torch.einsum("ir,jr,kr->ijk",U_o,V_o,W_o), T_target).item()
        
    return {"tid":tid,"col":col_drop,"mse_o":mse_o,"mse_t":best_loss,"ep":ep+1,"t":time.time()-start_t,"U":U_o,"V":V_o,"W":W_o}

print("Trial runner with Custom Match Callback ready")

Trial runner with Custom Match Callback ready


In [5]:
# --- UPDATED COLUMN SORTING (BYPASSING COL 21) ---
U_t = torch.tensor(U_23); V_t = torch.tensor(V_23); W_t = torch.tensor(W_23)
col_str = torch.norm(U_t,dim=0)*torch.norm(V_t,dim=0)*torch.norm(W_t,dim=0)

# Sort all columns by strength
all_sorted_cols = torch.argsort(col_str).tolist()

print("Column strength (weakest first, EXCLUDING Col 21):")
# Filter out 21 since we mathematically proved it creates a 0.017 floor
sorted_cols = [c for c in all_sorted_cols if c != 21]

for rk,c in enumerate(sorted_cols):
    print(f"  #{rk+1:2d}: Col {c:2d} | strength={col_str[c]:.2f}")

# Redistribute the trials favoring the true weak links (like Col 13)
trial_cols = []
for rk,c in enumerate(sorted_cols):
    trial_cols.extend([c]*max(1,3-rk//8))
    
trial_cols = (trial_cols*5)[:cfg.N_TRIALS]
print(f"New Distribution: {dict(sorted(Counter(trial_cols).items()))}")

Column strength (weakest first, EXCLUDING Col 21):
  # 1: Col 13 | strength=1.41
  # 2: Col 20 | strength=1.73
  # 3: Col 16 | strength=2.00
  # 4: Col 18 | strength=2.45
  # 5: Col  3 | strength=2.45
  # 6: Col  6 | strength=2.45
  # 7: Col  9 | strength=2.45
  # 8: Col 14 | strength=2.45
  # 9: Col 12 | strength=2.83
  #10: Col  0 | strength=3.00
  #11: Col  5 | strength=3.46
  #12: Col  8 | strength=3.46
  #13: Col 10 | strength=3.46
  #14: Col 15 | strength=4.24
  #15: Col  7 | strength=4.24
  #16: Col 22 | strength=4.47
  #17: Col 19 | strength=4.69
  #18: Col  4 | strength=4.90
  #19: Col  2 | strength=4.90
  #20: Col 11 | strength=4.90
  #21: Col 17 | strength=5.20
  #22: Col  1 | strength=6.48
New Distribution: {0: 2, 1: 1, 2: 1, 3: 3, 4: 1, 5: 2, 6: 3, 7: 2, 8: 2, 9: 3, 10: 2, 11: 1, 12: 2, 13: 6, 14: 3, 15: 2, 16: 3, 17: 1, 18: 3, 19: 1, 20: 4, 22: 2}


In [6]:
out = "outputs"
if not os.path.exists(out): os.makedirs(out)
print(f"\nBASIS RANDOMIZATION: {cfg.N_TRIALS} trials\n")
results = []; best_mse = float("inf"); best_sol = None; t0 = time.time()
for t in range(cfg.N_TRIALS):
    col = trial_cols[t]
    print(f"\n--- Trial {t+1}/{cfg.N_TRIALS} | Drop Col {col} ---")
    r = run_trial(t+1, col, verbose=True)
    mse = r["mse_o"]; results.append(r)
    tag = " ***NEW BEST***" if mse < best_mse else ""
    if mse < best_mse:
        best_mse = mse; best_sol = r
        if r["U"] is not None:
            torch.save({"U":r["U"].cpu(),"V":r["V"].cpu(),"W":r["W"].cpu(),"mse":mse}, os.path.join(out,"best_cont.pt"))
    print(f'  >>> DONE | MSE: {mse:.2e} | {r["t"]:.1f}s{tag}')
t1 = time.time()

print(f"\nDONE: {t1-t0:.0f}s | Best MSE: {best_mse:.2e}")


BASIS RANDOMIZATION: 50 trials


--- Trial 1/50 | Drop Col 13 ---
  T 1 Ep    0 MSE:1.65e-01 Noi:0.150
  T 1 Ep  500 MSE:5.67e+01 Noi:0.137
  T 1 Ep 1000 MSE:3.83e+01 Noi:0.125
  T 1 Ep 1500 MSE:4.56e+01 Noi:0.112
  T 1 Ep 2000 MSE:2.50e+01 Noi:0.100
  T 1 Ep 2500 MSE:2.02e+01 Noi:0.087
  T 1 Ep 3000 MSE:1.15e+01 Noi:0.080
  T 1 Ep 3500 MSE:2.53e+01 Noi:0.076
  T 1 Ep 4000 MSE:2.26e+01 Noi:0.072
  T 1 Ep 4500 MSE:1.50e+01 Noi:0.068
  T 1 Ep 5000 MSE:1.16e+01 Noi:0.064
  T 1 Ep 5500 MSE:8.25e+00 Noi:0.060
  T 1 Ep 6000 MSE:6.37e+00 Noi:0.056
  T 1 Ep 6500 MSE:4.42e+00 Noi:0.052
  T 1 Ep 7000 MSE:5.15e+00 Noi:0.048
  T 1 Ep 7500 MSE:3.62e+00 Noi:0.044
  T 1 Ep 8000 MSE:1.88e+00 Noi:0.020
  T 1 Ep 8500 MSE:1.64e+00 Noi:0.019
  T 1 Ep 9000 MSE:1.58e+00 Noi:0.018
  T 1 Ep 9500 MSE:1.31e+00 Noi:0.018
  T 1 Ep10000 MSE:9.53e-01 Noi:0.017
  T 1 Ep10500 MSE:7.40e-01 Noi:0.016
  T 1 Ep11000 MSE:6.33e-01 Noi:0.015
  T 1 Ep11500 MSE:4.48e-01 Noi:0.014
  T 1 Ep12000 MSE:3.24e-01 Noi:0.013
  T 1 Ep

In [10]:
def round_vocab(x, vt=VOCAB_T):
    vt = vt.to(x.device)
    return vt[torch.argmin(torch.abs(x.unsqueeze(-1)-vt.unsqueeze(0).unsqueeze(0)), dim=-1)]

def fine_tune(Ui,Vi,Wi, n_epochs=5000):
    U = Ui.float().clone().detach().requires_grad_(True)
    V = Vi.float().clone().detach().requires_grad_(True)
    W = Wi.float().clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([U,V,W], lr=0.001)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    best = float("inf"); best_s = None
    for ep in range(n_epochs):
        opt.zero_grad()
        mse = F.mse_loss(torch.einsum("ir,jr,kr->ijk",U,V,W), T_target)
        l1 = 0.001*(U.abs().mean()+V.abs().mean()+W.abs().mean())
        (mse+l1).backward(); opt.step(); sch.step()
        if mse.item() < best:
            best = mse.item(); best_s = (U.detach().clone(),V.detach().clone(),W.detach().clone())
        if best < 1e-8: break
    Uf,Vf,Wf = best_s
    Ur = round_vocab(Uf); Vr = round_vocab(Vf); Wr = round_vocab(Wf)
    mse_f = F.mse_loss(torch.einsum("ir,jr,kr->ijk",Ur.float(),Vr.float(),Wr.float()).cpu(), T_target.cpu()).item()
    return Ur,Vr,Wr,mse_f

if best_sol:
    bU,bV,bW = best_sol["U"].cpu(),best_sol["V"].cpu(),best_sol["W"].cpu()
    Ui = round_vocab(bU); Vi = round_vocab(bV); Wi = round_vocab(bW)
    Uf,Vf,Wf,mse_ft = fine_tune(Ui.to(DEVICE),Vi.to(DEVICE),Wi.to(DEVICE))
    print(f"After fine-tune MSE: {mse_ft:.2e}")
    if mse_ft < 1e-6: print("*** EXACT INTEGER RANK-22 SOLUTION! ***")
    elif mse_ft < 1e-4: print(f"Near-exact (MSE: {mse_ft:.2e})")
    else: print("Integer rounding degraded")
else:
    print("No best solution found.")

After fine-tune MSE: 2.74e-01
Integer rounding degraded


In [11]:
if best_sol:
    np.savez(os.path.join(out,"r22_factorization.npz"),U=Uf.cpu().numpy(),V=Vf.cpu().numpy(),W=Wf.cpu().numpy())
    print(f"Saved: {out}/r22_factorization.npz")

Saved: outputs/r22_factorization.npz
